# 02 ? Prepara??o dos dados
Continua??o da EDA, preservando o CSV original. Nesta primeira etapa verificamos os dados e congelamos a divis?o; a prepara??o das entradas ser? acrescentada de seguida.

In [1]:
from pathlib import Path
import sys
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/cybercrime_forensic_dataset.csv').exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.dados import carregar_dados, dividir_dados, resumo_divisao
df = carregar_dados()
print("Dimens?o:", df.shape)
df.info()
display(df.isna().sum().to_frame("Ausentes"))
display(df.nunique().to_frame("Valores distintos"))
print("Duplicados completos:", df.duplicated().sum())
display(df["Label"].value_counts())

Dimens?o: (7400, 11)
<class 'pandas.DataFrame'>
RangeIndex: 7400 entries, 0 to 7399
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Timestamp          7400 non-null   str    
 1   User_ID            7400 non-null   int64  
 2   IP_Address         7400 non-null   str    
 3   Activity_Type      7400 non-null   str    
 4   Resource_Accessed  7400 non-null   str    
 5   File_Name          3196 non-null   str    
 6   Action             7400 non-null   str    
 7   Login_Attempts     2105 non-null   float64
 8   File_Size          3196 non-null   float64
 9   Anomaly_Type       7400 non-null   str    
 10  Label              7400 non-null   str    
dtypes: float64(2), int64(1), str(8)
memory usage: 636.1 KB


,Ausentes
Timestamp,0
User_ID,0
IP_Address,0
Activity_Type,0
Resource_Accessed,0
File_Name,4204
Action,0
Login_Attempts,5295
File_Size,4204
Anomaly_Type,0


,Valores distintos
Timestamp,7400
User_ID,5031
IP_Address,7353
Activity_Type,7
Resource_Accessed,36
File_Name,6
Action,7
Login_Attempts,10
File_Size,2382
Anomaly_Type,5


Duplicados completos: 0


Label
Normal        6167
Suspicious    1233
Name: count, dtype: int64

## Divis?o por utilizador
Existem v?rios registos do mesmo utilizador. O objetivo experimental fica definido como classificar atividades de **utilizadores n?o observados no treino**, dentro da popula??o e per?odo simulados. GroupShuffleSplit, com semente 42, reserva 20% dos utilizadores para teste; dos restantes, 25% ficam para valida??o (aproximadamente 60%/20%/20% dos utilizadores no total).
As percentagens de linhas e classes podem diferir: n?o se estratificam os r?tulos nem se procura uma divis?o com melhores resultados. A semente n?o ser? ajustada em fun??o das m?tricas.
Uma divis?o por linhas permitiria utilizadores repetidos nos conjuntos; uma divis?o temporal responderia ? previs?o de per?odos futuros, que n?o ? a alega??o desta experi?ncia. N?o existe um identificador de incidente para garantir a separa??o de acontecimentos relacionados.
O teste j? foi abrangido pela EDA original; essa limita??o ? reconhecida. As contagens seguintes verificam apenas a composi??o e n?o ser?o utilizadas para selecionar modelos.

In [2]:
conjuntos = dividir_dados(df)
display(resumo_divisao(conjuntos))
for a, b in [("treino", "validacao"), ("treino", "teste"), ("validacao", "teste")]:
    assert set(conjuntos[a]["User_ID"]).isdisjoint(conjuntos[b]["User_ID"])
    assert set(conjuntos[a].index).isdisjoint(conjuntos[b].index)
assert sum(len(parte) for parte in conjuntos.values()) == len(df)
print("Divis?o congelada: sem sobreposi??o de linhas ou utilizadores.")

,Registos,Utilizadores,Normal,Suspicious,% Suspicious
Conjunto,,,,,
treino,4416,3018,3682,734,16.62
validacao,1478,1006,1237,241,16.31
teste,1506,1007,1248,258,17.13


Divis?o congelada: sem sobreposi??o de linhas ou utilizadores.
